In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to path
sys.path.append(str(Path("..").resolve()))

# Laden von trainierten Modellen


In [7]:
from helpers import load_fitted_model

model_configurations = [
    ("hmm", 2, 42),
    ("hmm", 3, 42),
    ("hmm", 4, 42),  # Konfigurationen mit Seed 42
    ("vdhmm", 2, 42),
    ("vdhmm", 3, 42),
    ("vdhmm", 4, 42),
    ("hmm", 2, 43),
    ("hmm", 3, 43),
    ("hmm", 4, 43),
    ("vdhmm", 2, 43),
    ("vdhmm", 3, 43),  # Konfigurationen mit Seed 43
    ("vdhmm", 4, 43),
    ("hmm", 2, 123),
    ("hmm", 3, 123),
    # ("hmm", 4, 123),  # Konfigrationen mit Seed 123
    ("vdhmm", 2, 123),
    ("vdhmm", 3, 123),
    # ("vdhmm", 4, 123),
    ("hmm", 2, None),  # Konfiguration mit Seed None (Original Indizes)
    ("hmm", 3, None),
    ("hmm", 4, None),
    ("vdhmm", 2, None),  # Konfiguration mit Seed None (Original Indizes)
    ("vdhmm", 3, None),
    ("vdhmm", 4, None),
]

n_models = len(model_configurations)

# get different metrics out of hmm models

results_df = pd.DataFrame(
    {
        "Model": [
            f"{model_name}_{S}" for (model_name, S, seed) in model_configurations
        ],
        "Seed": [seed for (model_name, S, seed) in model_configurations],
        # "LOOIC": [0.0] * n_models,
        # "WAIC": [0.0] * n_models,
        # "LPD_TRAIN": [0.0] * n_models,
        # "LPD_VAL": [0.0] * n_models,
        "AUC_TRAIN": [0.0] * n_models,
        "AUC_TEST": [0.0] * n_models,
        "R_HAT_MEAN": [0.0] * n_models,
        "R_HAT_STD": [0.0] * n_models,
    }
)

In [ ]:
import arviz as az
from sklearn.metrics import roc_auc_score
import warnings

# Unterdrücke ArviZ Warnungen und Output
warnings.filterwarnings("ignore")

# Schleife über alle Modell-Konfigurationen
for k, (model_name, S, seed) in enumerate(model_configurations):
    print(f"Processing {k+1}/{n_models}: {model_name} with {S} states (seed={seed})...")

    # get original eval indices for each seed
    np.random.seed(seed)
    indices_val_cal = np.random.permutation(np.arange(500, 921))

    # Lade das Modell mit Seed
    try:
        model = load_fitted_model(model_name, S, seed=seed)
    except FileNotFoundError:
        print(
            f"FileNotFound: {model_name} with {S} states and seed {seed} was not found! Skipping..."
        )
        continue

    log_lik = model.fit.stan_variable("log_lik")
    log_lik_test = model.fit.stan_variable("log_lik_test")
    close_prob = model.fit.stan_variable("close_prob")
    close_prob_mean = close_prob.mean(axis=0)

    idata = az.from_cmdstanpy(model.fit, log_likelihood="log_lik")

    # Berechne looc, waic, lpd (ohne Output)
    # results_df.loc[k, "LOOIC"] = az.loo(idata, var_name="log_lik").elpd_loo
    # results_df.loc[k, "WAIC"] = az.waic(idata, var_name="log_lik").elpd_waic
    # results_df.loc[k, "LPD_TRAIN"] = -2 * np.sum(np.log(np.exp(log_lik).mean(axis=0)))
    # results_df.loc[k, "LPD_VAL"] = -2 * np.sum(
    #     np.log(np.exp(log_lik_test).mean(axis=0))
    # )

    # Berechne AUC für Trainingsdaten (in Prozent)
    y_true_train = np.array(model.stan_data["Closed"][:500])
    results_df.loc[k, "AUC_TRAIN"] = 100 * roc_auc_score(
        y_true_train, close_prob_mean[:500]
    )

    y_true_test = np.array(model.stan_data["Closed"])[eval_indices]
    # Annahme: Die Schrittweite passt, ggf. muss die Dimensionalität geprüft werden!
    close_prob_test_mean = np.mean(model.fit.stan_variable("close_prob"), axis=0)
    results_df.loc[k, "AUC_TEST"] = 100 * roc_auc_score(
        y_true_test, close_prob_mean[eval_indices]
    )

    # Verwende das bereits gespeicherte summary DataFrame (bereits in model.summary!)
    # Berechne Durchschnitt und Standardabweichung von R_hat über alle Parameter
    results_df.loc[k, "R_HAT_MEAN"] = np.nanmean(model.summary["R_hat"])
    results_df.loc[k, "R_HAT_STD"] = np.nanstd(model.summary["R_hat"])

print("\nErgebnisse:")

# Ergebnisse runden
results_df["AUC_TRAIN"] = results_df["AUC_TRAIN"].round(2)
results_df["AUC_TEST"] = results_df["AUC_TEST"].round(2)
results_df["R_HAT_MEAN"] = results_df["R_HAT_MEAN"].round(4)
results_df["R_HAT_STD"] = results_df["R_HAT_STD"].round(4)


np.random.seed(None)

results_df

Processing 1/22: hmm with 2 states (seed=42)...


AssertionError: 